In [1]:
import numpy as np
import pandas as pd

class MyLogisticRegression:
    def __init__(self, learning_rate=0.1, max_iter=100000, tol=1e-3):
        self.learning_rate = learning_rate
        self.max_iter = max_iter
        self.tol = tol
        self.weights = None
        self.bias = None
        self.loss_history = []

    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def _compute_loss(self, X, y):
        n = len(y)
        y_pred = self._predict_proba(X)
        loss = -1/n * np.sum(y*np.log(y_pred) + (1-y)*np.log(1-y_pred))
        return loss

    def _predict_proba(self, X):
        return self._sigmoid(np.dot(X, self.weights) + self.bias)

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0.0
        prev_loss = float('inf')

        for i in range(self.max_iter):
            y_pred = self._predict_proba(X)
            dw = (1/n_samples) * np.dot(X.T, (y_pred - y))
            db = (1/n_samples) * np.sum(y_pred - y)

            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db

            current_loss = self._compute_loss(X, y)
            self.loss_history.append(current_loss)

            if abs(prev_loss - current_loss) < self.tol:
                print(f"迭代{i+1}次后收敛")
                break
            prev_loss = current_loss

    def predict(self, X):
        if self.weights is None:
            raise ValueError("模型未训练")
        proba = self._predict_proba(X)
        return np.where(proba >= 0.5, 1, 0)

def standardize(X_train, X_test):
    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)
    std[std == 0] = 1
    X_train_std = (X_train - mean) / std
    X_test_std = (X_test - mean) / std
    return X_train_std, X_test_std, mean, std

def accuracy(y_true, y_pred):
    return np.sum(y_true == y_pred) / len(y_true)

data = pd.read_csv("./data/红酒品质分类.csv")
X = data.iloc[:, :-1].values
data['label'] = (data['quality'] > 6).astype(int)
y=data['label'].values
total = X.shape[0]
train_size = int(total * 0.8)

x_train_raw = X[:train_size]
y_train = y[:train_size]
x_test_raw = X[train_size:]
y_test = y[train_size:]

x_train, x_test, _, _ = standardize(x_train_raw, x_test_raw)

estimator = MyLogisticRegression()
estimator.fit(x_train, y_train)

y_pred = estimator.predict(x_test)
score = accuracy(y_test, y_pred)
print("正确率：", score)




迭代74次后收敛
正确率： 0.915625
